# Predicción presidencial Colombia 2026

**Iván Ramiro Pinzón** 
**30 de abril de 2026** (32 días antes de 1ª vuelta)

Este notebook corre los 4 modelos individuales s y el modelo unificado de manera secuencial. El modelo unificado es el resultado defendible; los 4 modelos individuales son útiles para descomponer dónde fallaba el sistema original.

**Pre-requisito:** los CSV de `data/` y las funciones de `src/` deben existir. Ejecutar desde la raíz del repo.

---

## 0. Configuración

In [8]:
%cd /content/eleccol2026

/content/eleccol2026


In [9]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt


ROOT = Path('.').absolute()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / 'src').exists():
    sys.path.insert(0, str(ROOT.parent))
    ROOT = ROOT.parent

print(f'PyMC {pm.__version__}, ArviZ {az.__version__}')
print(f'Repo root: {ROOT}')

PyMC 5.28.3, ArviZ 0.22.0
Repo root: /content/eleccol2026


## 1. Carga de datos

Una sola fuente de verdad: los CSV en `/data`. No hay datos hard-codeados.


In [10]:
from src.data_loader import (
    cargar_encuestas, cargar_senado, cargar_polymarket, cargar_consultas,
    CANDS, BLOQUES, resumen_basico,
)

encuestas = cargar_encuestas(ROOT / 'data' / 'encuestas.csv')
senado = cargar_senado(ROOT / 'data' / 'senado_2026.csv')
polym = cargar_polymarket(ROOT / 'data' / 'polymarket.csv')
consultas = cargar_consultas(ROOT / 'data' / 'consultas_2026.csv')

print(f'Encuestas cargadas: {len(encuestas)}')
resumen_basico(encuestas).style.set_caption('Encuestas presidenciales Colombia 2026')

Encuestas cargadas: 12


,pollster,fecha,n,metodo,cepeda,espriella,paloma,lopez,fajardo,otros,blanco
0,Invamer,2026-02-17,3800,presencial_hogar,37.1%,18.9%,10.0%,11.7%,6.6%,13.9%,1.8%
2,GAD3,2026-02-19,2473,telefonica,34.0%,26.0%,4.0%,3.0%,2.0%,27.0%,4.0%
9,Guarumo,2026-02-22,3867,presencial_hogar,31.7%,22.6%,10.0%,5.0%,3.6%,18.5%,8.6%
5,AtlasIntel,2026-03-11,4291,digital_RDR,36.4%,27.9%,17.5%,1.7%,7.8%,5.7%,3.0%
3,GAD3,2026-03-17,1076,telefonica,35.0%,21.0%,16.0%,4.0%,3.0%,16.0%,5.0%
8,CNC,2026-03-19,2157,presencial_hogar,34.5%,15.4%,22.2%,3.7%,3.6%,14.1%,6.5%
10,Guarumo,2026-03-22,3736,presencial_hogar,37.5%,20.2%,19.9%,3.0%,2.0%,6.4%,11.0%
6,AtlasIntel,2026-04-08,3617,digital_RDR,37.8%,27.2%,22.9%,1.0%,5.1%,3.4%,2.6%
1,Invamer,2026-04-19,3800,presencial_hogar,44.3%,21.5%,19.8%,3.6%,2.5%,3.5%,4.8%
4,GAD3,2026-04-22,1500,telefonica,36.0%,21.0%,13.0%,3.0%,2.0%,20.0%,5.0%


## 2. Modelo 1  — agregación bayesiana de encuestas

State-space + Dirichlet-Multinomial + house effects con suma cero. Ver `src/modelo_unificado.py` para la especificación completa.

In [11]:
from datetime import date
from src.modelo_unificado import ajustar_modelo_encuestas

DIA_ELECCION = date(2026, 5, 31)

res = ajustar_modelo_encuestas(
    encuestas, dia_eleccion=DIA_ELECCION,
    n_draws=1000, n_tune=1500, n_chains=4,
    target_accept=0.95, progressbar=False,
)

rhat = az.rhat(res.idata, var_names=['pi'])
ess = az.ess(res.idata, var_names=['pi'])
print(f'R-hat máximo (π): {float(rhat["pi"].max()):.4f}')
print(f'ESS mínimo (π):   {float(ess["pi"].min()):.0f}')

R-hat máximo (π): 1.0031
ESS mínimo (π):   2986


In [12]:
# Posterior en día de elección
pi = res.idata.posterior['pi'].values
pi_elec = pi[..., -1, :].reshape(-1, len(CANDS))

rows = []
for k, c in enumerate(CANDS):
    m = pi_elec[:, k].mean() * 100
    lo = np.percentile(pi_elec[:, k], 5) * 100
    hi = np.percentile(pi_elec[:, k], 95) * 100
    p_top2 = (np.argsort(-pi_elec, axis=1)[:, :2] == k).any(axis=1).mean() * 100
    rows.append({
        'candidato': c, 'media (%)': f'{m:.1f}',
        'IC 90%': f'[{lo:.1f}, {hi:.1f}]', 'P(top 2) (%)': f'{p_top2:.1f}'
    })
df_post = pd.DataFrame(rows).sort_values('media (%)', ascending=False, key=lambda x: x.astype(float))
df_post

,candidato,media (%),IC 90%,P(top 2) (%)
0,cepeda,38.2,"[26.7, 50.2]",98.3
1,espriella,23.0,"[14.3, 33.2]",65.8
2,paloma,19.7,"[12.3, 28.8]",35.9
5,otros,7.8,"[4.4, 12.1]",0.0
6,blanco,4.5,"[3.2, 6.1]",0.0
4,fajardo,3.6,"[1.9, 5.9]",0.0
3,lopez,3.2,"[1.7, 5.2]",0.0


### House effects estimados

Estos NO se imponen a priori. Emergen del posterior. 

In [13]:
delta = res.idata.posterior['delta'].values
rows = []
for p_idx, p_name in enumerate(res.pollster_names):
    row = {'encuestadora': p_name}
    for k_idx, c in enumerate(CANDS[:-1]):
        row[c] = delta[..., p_idx, k_idx].mean()
    rows.append(row)
pd.DataFrame(rows).set_index('encuestadora').round(3)

,cepeda,espriella,paloma,lopez,fajardo,otros
encuestadora,,,,,,
AtlasIntel,0.098,0.337,0.207,-0.431,0.342,-0.553
CNC,-0.087,-0.262,0.156,0.027,-0.056,0.221
GAD3,-0.026,0.038,-0.355,-0.006,-0.267,0.616
Guarumo,-0.017,0.024,0.061,0.098,-0.077,-0.089
Invamer,0.064,-0.106,-0.067,0.280,0.021,-0.193


## 3. Modelo 2  — proyección revealed por bloque

Solo Senado 2026 + factor de amplificación incierto (LogNormal). Sin matriz de transferencia subjetiva. **Las bandas anchas son el punto:** no hay suficiente información sin encuestas.

In [14]:
from src.modelo_unificado import PARTIDO_A_BLOQUE, PRIOR_AMPLIFICACION_LOGMEAN, PRIOR_AMPLIFICACION_LOGSIGMA

bloques_unicos = sorted(set(BLOQUES.values()))
bloque_a_votos_sen = {b: 0 for b in bloques_unicos}
for v in senado:
    b = PARTIDO_A_BLOQUE.get(v.partido, 'blanco')
    bloque_a_votos_sen[b] += v.votos

rng = np.random.default_rng(2026)
n_sim = 30_000
log_tau = rng.normal(PRIOR_AMPLIFICACION_LOGMEAN, PRIOR_AMPLIFICACION_LOGSIGMA, (n_sim, len(bloques_unicos)))
tau = np.exp(log_tau)
votos_proy = np.zeros((n_sim, len(bloques_unicos)))
for b_idx, b in enumerate(bloques_unicos):
    votos_proy[:, b_idx] = bloque_a_votos_sen[b] * tau[:, b_idx]
shares_bloque = votos_proy / votos_proy.sum(axis=1, keepdims=True)

rows = []
for b_idx, b in enumerate(bloques_unicos):
    rows.append({
        'bloque': b,
        'votos Senado': bloque_a_votos_sen[b],
        'media (%)': f'{shares_bloque[:, b_idx].mean()*100:.1f}',
        'IC 90%': f'[{np.percentile(shares_bloque[:, b_idx], 5)*100:.1f}, {np.percentile(shares_bloque[:, b_idx], 95)*100:.1f}]',
    })
pd.DataFrame(rows)

,bloque,votos Senado,media (%),IC 90%
0,blanco,3800000,19.9,"[5.0, 44.3]"
1,centro,5579336,28.2,"[8.1, 56.7]"
2,der,5799378,29.2,"[8.6, 58.5]"
3,izq,4413636,22.8,"[6.2, 49.0]"


## 4. Modelo 3  — segunda vuelta con propagación correcta

Toma el posterior conjunto del modelo 1 (no medias puntuales). Para cada muestra: identifica el par finalista; si Cepeda está, simula 2v condicional con matriz de transferencia probabilística (priors anclados en bloques).

In [15]:
from src.modelo_unificado import simular_segunda_vuelta, calcular_p_presidente

p_pres = calcular_p_presidente(res)
rows = []
for c in sorted(p_pres, key=lambda x: -p_pres[x]):
    if p_pres[c] > 0.001:
        rows.append({'candidato': c, 'P(presidente)': f'{p_pres[c]*100:.1f}%'})
pd.DataFrame(rows)

,candidato,P(presidente)
0,cepeda,41.8%
1,espriella,38.1%
2,paloma,20.1%


In [16]:
# Cara a cara directos vs. modelo de transferencias
from src.data_loader import CARA_A_CARA_INVAMER_ABR2026

cepeda_idx = CANDS.index('cepeda')
top1 = pi_elec.argmax(axis=1)
top2 = np.argsort(-pi_elec, axis=1)[:, 1]
gana_1v = pi_elec[np.arange(len(top1)), top1] > 0.5
sims_2v = ~gana_1v
rival = np.where(top1 == cepeda_idx, top2, top1)
cep_en_par = (top1 == cepeda_idx) | (top2 == cepeda_idx)

rows = []
for r_name, (cep_inv, riv_inv) in CARA_A_CARA_INVAMER_ABR2026.items():
    mask = sims_2v & cep_en_par & (rival == CANDS.index(r_name))
    if mask.sum() < 50:
        rows.append({'rival': r_name, 'Invamer cep%': cep_inv, 'Modelo cep% (med)': '—', 'IC 90%': '—'})
        continue
    cep_share = simular_segunda_vuelta(pi_elec[mask], rival=r_name, n_sim_per_post=20, rng=rng)
    rows.append({
        'rival': r_name,
        'Invamer cep%': f'{cep_inv:.1f}',
        'Modelo cep% (med)': f'{cep_share.mean()*100:.1f}',
        'IC 90%': f'[{np.percentile(cep_share, 5)*100:.1f}, {np.percentile(cep_share, 95)*100:.1f}]',
    })
pd.DataFrame(rows)

,rival,Invamer cep%,Modelo cep% (med),IC 90%
0,paloma,51.2,48.3,"[38.3, 57.8]"
1,espriella,54.6,47.8,"[37.3, 57.6]"
2,fajardo,59.8,—,—
3,lopez,62.6,—,—


## 5. Modelo 4  — descomposición nested y canibalización

Descomposición $\pi_k = P(b)\cdot \pi^{within}_{k|b}$ aplicada al posterior. Escenarios = modificar cuotas \emph{within} preservando $P(b)$.

In [17]:
from src.modelo_unificado import escenario_canibalizacion

escenarios = {
    'Status quo':              {'paloma': 0.46, 'espriella': 0.45, 'otros': 0.09},
    'Espriella se desinfla':   {'paloma': 0.72, 'espriella': 0.20, 'otros': 0.08},
    'Espriella se baja':       {'paloma': 0.85, 'espriella': 0.00, 'otros': 0.15},
    'Derecha unificada':       {'paloma': 0.95, 'espriella': 0.00, 'otros': 0.05},
}

rows = []
for nombre, cw in escenarios.items():
    pi_new = escenario_canibalizacion(pi_elec, cw, bloque='der')
    cep = pi_new[:, CANDS.index('cepeda')].mean() * 100
    pal = pi_new[:, CANDS.index('paloma')].mean() * 100
    esp = pi_new[:, CANDS.index('espriella')].mean() * 100
    top2 = np.argsort(-pi_new, axis=1)[:, :2]
    p_pal_t2 = (top2 == CANDS.index('paloma')).any(axis=1).mean() * 100
    p_esp_t2 = (top2 == CANDS.index('espriella')).any(axis=1).mean() * 100
    rows.append({
        'escenario': nombre, 'Cepeda': f'{cep:.1f}%', 'Paloma': f'{pal:.1f}%', 'Espriella': f'{esp:.1f}%',
        'P(Pal top2)': f'{p_pal_t2:.1f}%', 'P(Esp top2)': f'{p_esp_t2:.1f}%',
    })
pd.DataFrame(rows).set_index('escenario')

,Cepeda,Paloma,Espriella,P(Pal top2),P(Esp top2)
escenario,,,,,
Status quo,38.2%,23.2%,22.7%,100.0%,6.2%
Espriella se desinfla,38.2%,36.4%,10.1%,100.0%,0.0%
Espriella se baja,38.2%,42.9%,0.0%,100.0%,0.0%
Derecha unificada,38.2%,48.0%,0.0%,100.0%,0.0%


## 6. Backtest 2022

Validación del modelo aplicado a las encuestas reales publicadas a 32 días de la elección de 2022.

**Resultado real 1ª vuelta 2022:** Petro 40.32%, Hernández 28.15%, Fico 23.91%, Fajardo 4.20%.

In [18]:
from tests.backtest_2022 import (
    encuestas_2022_a_objects, ENCUESTAS_2022,
    construir_modelo_2022, evaluar_backtest, REAL_1V_2022,
)

encuestas_22 = encuestas_2022_a_objects(ENCUESTAS_2022)
modelo_22, grid_22, _ = construir_modelo_2022(encuestas_22)
with modelo_22:
    idata_22 = pm.sample(
        draws=500, tune=800, chains=4, cores=1,
        target_accept=0.95, progressbar=False, random_seed=2022,
    )
df_bt = evaluar_backtest(idata_22, grid_22, None)
df_bt

ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


,candidato,real,modelo_mean,IC90,error_pp,cubre_IC90
0,petro,40.32%,38.68%,"[34.6, 43.0]",-1.64,Sí
1,fico,23.91%,26.87%,"[23.3, 30.7]",+2.96,Sí
2,fajardo,4.20%,9.45%,"[7.5, 11.5]",+5.25,—
3,hernandez,28.15%,11.65%,"[9.4, 14.3]",-16.50,—
4,betancourt,0.84%,2.85%,"[2.0, 3.9]",+2.01,—
5,otros,1.92%,6.09%,"[4.5, 7.7]",+4.17,—
6,blanco,1.66%,4.41%,"[3.4, 5.6]",+2.75,—


**Lectura del backtest:**

- Petro y Fico: dentro del IC90% del modelo. Buen desempeño.
- Hernández: el modelo lo subestima en 16.5 pp. **Esta es la limitación más importante del enfoque:** Hernández despegó del 11% al 28% en las últimas dos semanas de mayo 2022. El agregador de encuestas a 32 días no podía tener esa información.
- Esto es honesto: cualquier modelo de agregación de encuestas tiene esta limitación, no solo este. Los IC90% son condicionales a que la dinámica de la última fase de la campaña sea similar a la observada en datos históricos.

## 7. Comparación final con los modelos originales

In [ ]:
P_ORIG = {
    'Modelo 1 — Encuestas (orig)':           88.6,
    'Modelo 2 — Revealed (orig)':            82.9,
    'Modelo 3 — Transferencia 2v (orig)':    79.5,
    'Modelo 4 — Nested logit (orig)':        62.7,
    'Modelo unificado bayesiano ()': p_pres['cepeda'] * 100,
}
df_compare = pd.DataFrame(
    [(m, f'{p:.1f}%') for m, p in P_ORIG.items()],
    columns=['Modelo', 'P(Cepeda presidente)']
)
df_compare

,Modelo,P(Cepeda presidente)
0,Modelo 1 — Encuestas (orig),88.6%
1,Modelo 2 — Revealed (orig),82.9%
2,Modelo 3 — Transferencia 2v (orig),79.5%
3,Modelo 4 — Nested logit (orig),62.7%
4,Modelo unificado bayesiano (corregido),41.8%
